Testcase for the MTPCL implementation

In [1]:
# imports
import numpy as np
import gym_electric_motor as gem
from gym_electric_motor.physical_systems import ConstantSpeedLoad
from gym_electric_motor.reference_generators import  ConstReferenceGenerator
from gym_electric_motor.physical_system_wrappers import DeadTimeProcessor
from gym_electric_motor.physical_systems.solvers import EulerSolver

import gem_controllers as gc

In [ ]:
# define the environment

motor_env_id = "Cont-TC-EESM-v0"
tau = 1e-4    # The duration of each sampling step



T_generator = ConstReferenceGenerator('torque', 0.5)

motor_parameter = {
        'p': 2,
        'l_d': 3.78e-3,
        'l_q': 1.21e-3,
        'l_m': 40e-3, 
        'l_e': 870e-3,
        'j_rotor': 0.3883,
        'r_s': 123e-3,
        'r_e': 15.6,
        'k': 0.057,
    }

limit_values = dict(
    omega = 7e+3*np.pi/30,
    i = 31.25,
    u = 200,
    #u_e = 200,
    i_e = 6.25,
    torque = 20,
    )

nominal_values=dict(
    omega = 2e+3*np.pi/30,  # angular velocity in rad/s
    i = 25,                   # motor current in amps
    u = 200,                  # nominal voltage in volts
    #u_e = 200,
    i_e = 5,
    torque = 15,
    )

eesm_init = {
    'states': {
        'i_sd' : 0.,
        'i_sq' : 0.,
        'i_e' : 0.,
        'epsilon' : 0.,
        }
    }

load = ConstantSpeedLoad(omega_fixed=0.15*limit_values['omega'],)

physical_system_wrappers = [
    # Wrapped directly around the physical system
    #CosSinProcessor(angle='epsilon'),
    #DqToAbcActionProcessor.make('EESM'),
    DeadTimeProcessor(steps=1) # Only use DeadTimeWrapper after you have implemented a last action concatinator for the state
    # Wrapped around the CosSinProcessor. Therefore, the generated states (cos and sin) can be accessed.
]


env = gem.make(  
    motor_env_id,
    # visualize the results
    # parameterize the PMSM and update limitations
    motor=dict(
        motor_parameter=motor_parameter,
        limit_values=limit_values,
        nominal_values=nominal_values,
        motor_initializer=eesm_init,
    ),
    # define the random initialisation for load and motor
    load=load,
    tau=tau,
    ode_solver=EulerSolver(),
    physical_system_wrappers=physical_system_wrappers, # Pass the Physical System Wrappers
    # state_filter=["i_sd", "i_sq", "omega", "i_e", "torque", "cos(epsilon)", "sin(epsilon)"], # this is connected to following index
    #state_filter=["i_sd", "i_sq", "omega", "i_e", "cos(epsilon)", "sin(epsilon)"],
    supply=dict(u_nominal=200),
    #reference_generator = T_generator,
    #reward_function = reward_function
    )


# Initialize the controller
c = gc.GemController.make(
        env,
        motor_env_id,
        a=8,
        block_diagram=False,
        current_safety_margin=0.15,
        save_block_diagram_as=(),
    )



(obs,ref), _ = env.reset(seed = 1)
c.reset()

# Take trained agent and loop over environment
i_sd = []
i_sq = []
omegas = []
i_f = []
torque = []
torque_ref = []
action_stored = []
rewards_stored = []
# obs = ["i_sd", "i_sq", "omega", "i_e", "cos(epsilon)", "sin(epsilon)"], Tref, 3 actions

niveau0 = 0
niveau1 = -15/limit_values['torque'] 
niveau2 = 15/limit_values['torque'] 



i_sd_idx = env.get_wrapper_attr('physical_system').state_positions['i_sd']
i_sq_idx = env.get_wrapper_attr('physical_system').state_positions['i_sq']
omega_idx = env.get_wrapper_attr('physical_system').state_positions['omega']
i_f_idx = env.get_wrapper_attr('physical_system').state_positions['i_e']
torque_idx = env.get_wrapper_attr('physical_system').state_positions['torque']

In [ ]:
for i in range(65000):

    

    if i <= 2500:
        env.env.unwrapped.reference_generator._reference_value = niveau0        
    elif i <= 62500:
        env.env.unwrapped.reference_generator._reference_value = niveau1 + (niveau2 - niveau1) * (i - 2500) / 60000
    else:
        env.env.unwrapped.reference_generator._reference_value = niveau0
    ref_x = env.env.unwrapped.reference_generator._reference_value


    action = c.control(obs,ref_x)

    (obs, ref), rewards, terminated, truncated, info = env.step(action)
    done = terminated or truncated


    # TODO: record the current references of the look up table 

    i_sd.append(obs[i_sd_idx])
    i_sq.append(obs[i_sq_idx])
    omegas.append(obs[omega_idx])
    i_f.append(obs[i_f_idx])
    torque.append(obs[torque_idx])
    torque_ref.append(ref_x)
    action_stored.append(action)
    

    if done:
        #obs,_ = env.reset(seed = 1)
        break

: 

In [ ]:
from scipy.optimize import minimize
from scipy.interpolate import interp1d

In [ ]:
# %% Copper Losses Optimization for T_ref with specific speed
# Define the losses equation (to be minimized)
def losses_eq(input):
    isd, isq, i_f = input
    return (1.5 * (isd**2 + isq**2) * 123e-3) + (i_f**2 * 15.6)

# Define the constraints, 80persecond = 160 in ele
def constraints(torque_):
    return [
        {'type': 'ineq', 'fun': lambda input: 25 - np.sqrt(input[0]**2 + input[1]**2)},  # id**2 + iq**2 <= iLim**2
        {'type': 'ineq', 'fun': lambda input: 5**2 - input[2]**2},  # iF <= iLim
        {'type': 'ineq', 'fun': lambda input: (200/40)**2 - (1.21e-3 * input[1])**2 - (40e-3 * input[2] + 3.78e-3 * input[0])**2},  # voltage constraints
        # {'type': 'ineq', 'fun': lambda input: (100)**2 - 0},
        {'type': 'eq', 'fun': lambda input: ((1.5 * 2 * input[1]) * (40e-3 * input[2] + (3.78e-3 - 1.21e-3) * input[0])) - torque_}  # torque constraint
    ]

# Initial guess for the variables
initial_guess = [1, 1, 1]  # Start with non-zero initial guess to avoid division by zero in constraints

torque_range = np.multiply(torque_ref,limit_values['torque'])#np.linspace(-15, 15, 300)
ele_speed = 0.15 * limit_values['omega'] * 2 # Example electrical speed in rad/s

losses_Tmin = []

# Iterate over omega_el and torque values and perform optimization
for torque_ in torque_range:
    # Define the optimization problem for the current omega_el and torque
    result = minimize(losses_eq, initial_guess, constraints=constraints(torque_), method='SLSQP')
    
    # Extract the results
    if result.success:
        isd_opt, isq_opt, iF_opt = result.x
        minimized_value = result.fun  # This is the minimized losses value
        losses_Tmin.append((ele_speed, torque_, isd_opt, isq_opt, iF_opt, minimized_value))
    else:
        print(f"Optimization failed for omega_el={ele_speed} and torque={torque_}")
        losses_Tmin.append((ele_speed, torque_, None, None, None, None))

In [ ]:
id_MTPCL = np.zeros(len(losses_Tmin))
iq_MTPCL= np.zeros(len(losses_Tmin))
if_MTPCL = np.zeros(len(losses_Tmin))
CuLoss_160 = np.zeros(len(losses_Tmin))
t_MTPCL = np.zeros(len(losses_Tmin))
for i in range(len(losses_Tmin)):
    t_MTPCL[i] = losses_Tmin[i][1]
    if (losses_Tmin[i][2] == None):
        id_MTPCL[i] = id_MTPCL[i-1] 
        iq_MTPCL[i] = iq_MTPCL[i-1]
        if_MTPCL[i] = if_MTPCL[i-1]
        CuLoss_160[i] = CuLoss_160[i-1]
    else:
        id_MTPCL[i] = abs(losses_Tmin[i][2])
        iq_MTPCL[i] = losses_Tmin[i][3]
        if_MTPCL[i] = abs(losses_Tmin[i][4])
        CuLoss_160[i] = losses_Tmin[i][-1]
    
for j in range(len(losses_Tmin)):
    if (t_MTPCL[j] <= 0):
        iq_MTPCL[j] = -iq_MTPCL[j]
    else:
        iq_MTPCL[j] = iq_MTPCL[j]